# DL_03: BioCause Directed Relation Classification with PubMedBERT
**Causal NLP Reasoning — Phase 2: Deep Learning**

---

## 1. Purpose and Scope
This notebook implements fine-tuning of **PubMedBERT Fulltext** (`microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext`) for candidate-conditioned directed causal relation classification on the **BioCause** corpus.

Key requirements:
- Predict directed causal relations ($1 = \text{ARG1 causes ARG2}$, $0 = \text{ARG1 does not cause ARG2 under candidate convention}$).
- Retain explicit argument boundary markers: `[E1]...[/E1]` for Arg1 and `[E2]...[/E2]` for Arg2 in contextual text.
- Standardize on checkpoint `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext`.
- Handle class imbalance using dynamically computed training loss weights ($w_{\text{pos}} = 9,307 / 621 \approx 14.9871$). Never hardcode or derive weights from validation/test sets.
- Use conservative GPU defaults: Batch size 8, gradient accumulation 1, FP16 when supported on CUDA, `eval_strategy='epoch'`.
- Generate predictions for **both** validation and test splits:
  - `data/processed/dl_biocause_validation_predictions.csv` (1,884 rows)
  - `data/processed/dl_biocause_test_predictions.csv` (1,394 rows)
- Include required columns: `candidate_id, article_id, section_id, arg1_id, arg2_id, true_label, dl_prob, dl_pred, split`.
- Preserve raw predicted probabilities; use validation split strictly for decision threshold tuning; never use test labels for tuning.


## 2. Experimental Methodology & Theoretical Background

### Biomedical Domain Pre-training (PubMedBERT Fulltext)
- **PubMedBERT Fulltext** is pretrained *from scratch* on 14M PubMed abstracts and full-text PMC articles with a domain-specific vocabulary.
- It consistently outperforms general-domain BERT on biomedical relation extraction benchmarks.

### Directional Candidate Semantics & Class Imbalance
- **Candidate Generation**: All co-occurring entity pairs within sentences are extracted as directed candidates $(Arg_1, Arg_2)$.
- **Label Semantics**: Label `1` represents directed causation $Arg_1 \to Arg_2$. Label `0` indicates that $Arg_1$ does not cause $Arg_2$ in that candidate configuration. Label `0` includes reversed directional candidates $(Arg_2, Arg_1)$ and non-causal pairs.
- **Dynamic Imbalance Mitigation**: Model-eligible dataset preparation produces 9,928 training pairs (621 positive, 9,307 negative), giving an actual training ratio of **14.99:1**.
- The loss function incorporates a dynamic positive class weight: $w_{\text{pos}} = \frac{N_{\text{train, neg}}}{N_{\text{train, pos}}} = \frac{9307}{621} \approx 14.9871$.


## 3. Expected Inputs

| Split | Source File | Articles | Gold Relations | Model-Eligible Candidates | Neg / Pos Ratio |
| :--- | :--- | :---: | :---: | :---: | :---: |
| **Train** | `data/processed/dl_biocause_train.csv` | 12 PMC Articles | 621 | 9,928 | 14.99 : 1 |
| **Validation** | `data/processed/dl_biocause_val.csv` | 4 PMC Articles | 131 | 1,884 | 13.38 : 1 |
| **Test** | `data/processed/dl_biocause_test.csv` | 4 PMC Articles | 100 | 1,394 | 12.94 : 1 |


## 4. Technical Responsibilities & Implementation Requirements

1. **Portable Root Setup**: Resolve repository path dynamically across environments.
2. **Checkpoint & Tokenizer Configuration**: Standardize on `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` and register special tokens `['[E1]', '[/E1]', '[E2]', '[/E2]']`.
3. **Embedding Resizing**: Expand model token embeddings to accommodate newly added boundary tokens.
4. **Dynamic Weighted Loss**: Calculate $w_{\text{pos}}$ dynamically from training counts and configure weighted cross-entropy loss.
5. **Conservative Training Configuration**: Batch size 8, gradient accumulation 1, FP16 enabled on CUDA, `eval_strategy='epoch'`.
6. **Validation Prediction & Decision Threshold Tuning**: Predict probabilities on the 4 validation articles (1,884 candidates) and tune decision threshold $\tau^*$ to optimize validation F1.
7. **Held-Out Test Set Prediction**: Predict probabilities on the 1,394 test candidates using frozen $\tau^*$.
8. **Export Dual Prediction Artifacts**: Save both validation and test predictions to `data/processed/` for hybrid integration in `DL_04`.


In [3]:
# Section 1: Environment Setup & Portable Imports
import sys
import os
import json
import warnings
import re
from pathlib import Path

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Portability setup across Local repo, Colab, and Kaggle
def find_repo_root():
    curr = Path(os.getcwd()).resolve()
    preferred_roots = [Path('/content/causal-nlp-reasoning'), Path('/workspace/causal-nlp-reasoning'), Path('/kaggle/working/causal-nlp-reasoning')]
    search_roots = [*preferred_roots, curr, Path('/content'), Path('/workspace'), Path('/kaggle/working')]
    for base in search_roots:
        if not base.exists():
            continue
        candidates = [base, *base.parents]
        if base in {Path('/content'), Path('/workspace'), Path('/kaggle/working')}:
            candidates.extend(path.parent.parent for path in base.rglob('DL/dl_utils.py'))
        for candidate in candidates:
            if ((candidate / 'AML' / 'causal_extractors.py').is_file()
                    and (candidate / 'DL' / 'dl_utils.py').is_file()):
                return candidate.resolve()
    raise FileNotFoundError(
        'Repository root not found. Make the project folder available in the notebook workspace.'
    )
ROOT_DIR = find_repo_root()
for p in [ROOT_DIR, ROOT_DIR / 'AML', ROOT_DIR / 'DL']:
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

import dl_utils

# Enforce global determinism
SEED = 42
dl_utils.set_seed(SEED)

In [4]:
# Section 2: Configurable Hyperparameters, Hardware Detection & Dataset Ingestion
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, DataCollatorWithPadding
import numpy as np
import pandas as pd

PUBMEDBERT_CHECKPOINT = dl_utils.PUBMEDBERT_CHECKPOINT
assert PUBMEDBERT_CHECKPOINT == 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
device = dl_utils.get_device()
cuda_available = torch.cuda.is_available()
mps_available = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
FP16_ENABLED = bool(cuda_available and not mps_available)

BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 1
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 3
MAX_LENGTH = 384
EVAL_STRATEGY = 'epoch'
SAVE_TOTAL_LIMIT = 1
OUTPUT_DIR = str(ROOT_DIR / 'DL' / 'checkpoints' / 'biocause_pubmedbert')
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

DATA_DIR = ROOT_DIR / 'data' / 'processed'

def resolve_split_path(filename):
    candidates = [DATA_DIR / filename, ROOT_DIR / 'data' / 'processed' / filename]
    if Path('/content').exists():
        candidates.extend(Path('/content').rglob(filename))
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return DATA_DIR / filename

PRETRAIN_SPLIT_FILES = {
    'train': resolve_split_path('dl_biocause_train.csv'),
    'validation': resolve_split_path('dl_biocause_val.csv'),
}
REQUIRED_COLUMNS = [
    'candidate_id', 'article_id', 'section_id', 'arg1_id', 'arg2_id',
    'marked_text', 'label', 'split'
]
TARGET_COUNTS = dl_utils.BIOCAUSE_MODEL_ELIGIBLE_COUNTS

AUTHORITATIVE_VAL_CANDIDATE_IDS = pd.read_csv(
    PRETRAIN_SPLIT_FILES['validation'], usecols=['candidate_id']
)['candidate_id'].tolist()

dfs = {}
for split_name, split_path in PRETRAIN_SPLIT_FILES.items():
    if not split_path.exists():
        raise FileNotFoundError(f'Missing prepared BioCause {split_name} split.')
    df = pd.read_csv(split_path)
    dl_utils.validate_dataset_schema(df, REQUIRED_COLUMNS, dataset_name=f'BioCause-{split_name}')
    assert 'arg1_before_arg2' in df.columns, f'{split_name}: missing arg1_before_arg2 column'
    assert set(df['label'].unique()) == {0, 1}, f'{split_name}: both labels {{0, 1}} must occur'
    assert not df['marked_text'].isna().any() and not df['label'].isna().any()
    for marker in dl_utils.ENTITY_MARKERS:
        assert df['marked_text'].str.contains(re.escape(marker)).all(), (
            f'{split_name}: missing marker {marker}'
        )
    assert all(
        row.find('[E1]') < row.find('[/E1]') and row[row.find('[E1]') + 4:row.find('[/E1]')].strip() != ''
        and row.find('[E2]') < row.find('[/E2]') and row[row.find('[E2]') + 4:row.find('[/E2]')].strip() != ''
        for row in df['marked_text']
    ), f'{split_name}: empty or malformed entity span'
    expected = TARGET_COUNTS[split_name]
    assert len(df) == expected['total']
    assert int((df['label'] == 1).sum()) == expected['positive']
    assert int((df['label'] == 0).sum()) == expected['negative']
    assert df['candidate_id'].is_unique
    dfs[split_name] = df

train_df = dfs['train']
val_df = dfs['validation']
assert set(train_df['article_id']).isdisjoint(set(val_df['article_id']))
n_pos_train = int((train_df['label'] == 1).sum())
n_neg_train = int((train_df['label'] == 0).sum())
pos_weight = dl_utils.calculate_dynamic_class_weight(n_negative=n_neg_train, n_positive=n_pos_train)
assert np.isclose(pos_weight, 9307 / 621)
assert n_pos_train > 0 and n_neg_train > 0

split_summary_df = pd.DataFrame([
    {
        'Split': split_name,
        'Articles': int(df['article_id'].nunique()),
        'Candidates': int(len(df)),
        'Positive_1': int((df['label'] == 1).sum()),
        'Negative_0': int((df['label'] == 0).sum()),
        'Pos_Class_Weight': round(pos_weight, 4) if split_name == 'train' else '-',
    }
    for split_name, df in [('train', train_df), ('validation', val_df)]
])
split_summary_df

,Split,Articles,Candidates,Positive_1,Negative_0,Pos_Class_Weight
0,train,12,9928,621,9307,14.9871
1,validation,4,1884,131,1753,-


In [5]:
# Section 3: Tokenizer Setup & Dynamic Padding
tokenizer = dl_utils.prepare_tokenizer_and_inputs(
    checkpoint_name=PUBMEDBERT_CHECKPOINT,
    max_length=MAX_LENGTH,
    additional_special_tokens=dl_utils.ENTITY_MARKERS,
    token=False,
)

marker_token_ids = {
    marker: tokenizer.convert_tokens_to_ids(marker)
    for marker in dl_utils.ENTITY_MARKERS
}
assert len(marker_token_ids) == 4
assert all(
    token_id is not None and token_id != tokenizer.unk_token_id
    for token_id in marker_token_ids.values()
), f'Unregistered entity markers: {marker_token_ids}'

class BioCauseDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        encoded = tokenizer(
            list(texts),
            max_length=max_length,
            truncation=True,
        )
        self._input_ids = encoded['input_ids']
        self._attention_mask = encoded['attention_mask']
        self._labels = [int(y) for y in labels]

    def __len__(self):
        return len(self._labels)

    def __getitem__(self, idx):
        if isinstance(idx, str):
            if idx == 'input_ids':
                return self._input_ids
            if idx == 'attention_mask':
                return self._attention_mask
            if idx == 'labels':
                return self._labels
            raise KeyError(idx)
        return {
            'input_ids': self._input_ids[idx],
            'attention_mask': self._attention_mask[idx],
            'labels': self._labels[idx],
        }

train_tokenized = BioCauseDataset(train_df['marked_text'], train_df['label'], tokenizer, MAX_LENGTH)
val_tokenized = BioCauseDataset(val_df['marked_text'], val_df['label'], tokenizer, MAX_LENGTH)

required_marker_ids = list(marker_token_ids.values())
for split_name, dataset, exp_len in [
    ('train', train_tokenized, TARGET_COUNTS['train']['total']),
    ('validation', val_tokenized, TARGET_COUNTS['validation']['total']),
]:
    assert len(dataset) == exp_len, f'{split_name}: length {len(dataset)} != {exp_len}'
    assert max(map(len, dataset['input_ids'])) <= MAX_LENGTH, f'{split_name}: sequence exceeds {MAX_LENGTH}'
    assert set(dataset['labels']) == {0, 1}, f'{split_name}: invalid labels'
    assert all(all(mid in seq for mid in required_marker_ids) for seq in dataset['input_ids']), (
        f'{split_name}: lost entity marker token(s) after tokenization'
    )

In [6]:
# Section 4: Model Instantiation & Embedding Resize
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        PUBMEDBERT_CHECKPOINT, num_labels=2, local_files_only=True
    )
except Exception:
    model = AutoModelForSequenceClassification.from_pretrained(
        PUBMEDBERT_CHECKPOINT, num_labels=2, token=False
    )
dl_utils.resize_model_embeddings_for_tokenizer(model, tokenizer)
assert model.config.num_labels == 2
cls_head = model.classifier.out_proj if hasattr(model.classifier, 'out_proj') else model.classifier
assert cls_head.out_features == 2
assert model.get_input_embeddings().weight.shape[0] == len(tokenizer)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect ide

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [7]:
# Section 5: Correctly Normalized Weighted Loss & Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import inspect
import math

class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs['labels']
        model_inputs = {k: v for k, v in inputs.items() if k != 'labels'}
        outputs = model(**model_inputs)
        logits = outputs.logits
        assert logits.shape[-1] == 2
        weights = self.class_weights.to(device=logits.device, dtype=logits.dtype)
        loss = torch.nn.functional.cross_entropy(
            logits, labels, weight=weights, reduction='mean'
        )
        return (loss, outputs) if return_outputs else loss

def compute_metrics_with_classes(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    assert logits.shape[-1] == 2
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, labels=[0, 1], zero_division=0
    )
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision_0': precision[0], 'recall_0': recall[0], 'f1_0': f1[0],
        'precision_1': precision[1], 'recall_1': recall[1], 'f1_1': f1[1],
        'f1': float(np.mean(f1)),
    }

class_weights_tensor = torch.tensor([1.0, pos_weight], dtype=torch.float32)
training_args = dl_utils.get_training_arguments(
    output_dir=OUTPUT_DIR, num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS, learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY, fp16=FP16_ENABLED, eval_strategy=EVAL_STRATEGY,
    save_total_limit=SAVE_TOTAL_LIMIT,
)
training_args.per_device_eval_batch_size = EVAL_BATCH_SIZE
training_args.max_grad_norm = 1.0
total_train_steps = NUM_EPOCHS * max(1, math.ceil(len(train_tokenized) / (BATCH_SIZE * GRAD_ACCUM_STEPS)))
expected_warmup_steps = math.ceil(0.1 * total_train_steps)
training_args.warmup_steps = expected_warmup_steps
assert training_args.get_warmup_steps(total_train_steps) == expected_warmup_steps

trainer_kwargs = {
    'model': model, 'args': training_args, 'train_dataset': train_tokenized,
    'eval_dataset': val_tokenized, 'data_collator': data_collator,
    'compute_metrics': compute_metrics_with_classes,
    'class_weights': class_weights_tensor,
}
signature = inspect.signature(Trainer.__init__)
if 'processing_class' in signature.parameters:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in signature.parameters:
    trainer_kwargs['tokenizer'] = tokenizer
trainer = WeightedLossTrainer(**trainer_kwargs)
assert training_args.learning_rate == 1e-5
assert training_args.gradient_accumulation_steps == 1
assert training_args.max_grad_norm == 1.0
assert training_args.lr_scheduler_type == 'linear'
assert training_args.load_best_model_at_end is True
assert training_args.metric_for_best_model == 'eval_f1'
assert training_args.greater_is_better is True

In [8]:
# Section 6: Training, Validation Logging & Checkpoint Selection
TRAINING_COMPLETED = False
train_result = trainer.train()
best_ckpt = trainer.state.best_model_checkpoint
best_metric = trainer.state.best_metric
if not best_ckpt:
    raise RuntimeError('No validation-selected checkpoint was produced.')
assert np.isfinite(float(best_metric))
TRAINING_COMPLETED = True

epoch_metrics = []
latest_train_loss = None
for entry in trainer.state.log_history:
    if 'loss' in entry and 'eval_f1' not in entry:
        latest_train_loss = float(entry['loss'])
    if 'eval_f1' in entry:
        epoch_metrics.append({
            'epoch': float(entry['epoch']) if entry.get('epoch') is not None else float(len(epoch_metrics) + 1),
            'train_loss': latest_train_loss,
            'validation_loss': float(entry['eval_loss']) if entry.get('eval_loss') is not None else None,
            'validation_accuracy': float(entry['eval_accuracy']) if entry.get('eval_accuracy') is not None else None,
            'precision_0': float(entry['eval_precision_0']) if entry.get('eval_precision_0') is not None else None,
            'recall_0': float(entry['eval_recall_0']) if entry.get('eval_recall_0') is not None else None,
            'f1_0': float(entry['eval_f1_0']) if entry.get('eval_f1_0') is not None else None,
            'precision_1': float(entry['eval_precision_1']) if entry.get('eval_precision_1') is not None else None,
            'recall_1': float(entry['eval_recall_1']) if entry.get('eval_recall_1') is not None else None,
            'f1_1': float(entry['eval_f1_1']) if entry.get('eval_f1_1') is not None else None,
            'validation_macro_f1': float(entry['eval_f1']),
        })
assert len(epoch_metrics) == NUM_EPOCHS, f'Expected {NUM_EPOCHS} epoch evaluations, got {len(epoch_metrics)}'

selected_model = AutoModelForSequenceClassification.from_pretrained(best_ckpt, local_files_only=True)
selected_model.to(device)
assert selected_model.config.num_labels == 2
assert selected_model.get_input_embeddings().weight.shape[0] == len(tokenizer)
selected_head = selected_model.classifier.out_proj if hasattr(selected_model.classifier, 'out_proj') else selected_model.classifier
assert selected_head.out_features == 2
assert torch.isfinite(selected_head.weight).all() and torch.isfinite(selected_head.bias).all()
trainer.model = selected_model

epoch_metrics_df = pd.DataFrame(epoch_metrics)
epoch_metrics_df

Epoch,Training Loss,Validation Loss,Accuracy,Precision 0,Recall 0,F1 0,Precision 1,Recall 1,F1 1,F1
1,1.085094,1.503218,0.930467,0.930467,1.000000,0.963981,0.000000,0.000000,0.000000,0.481991
2,1.082808,1.447987,0.930467,0.930467,1.000000,0.963981,0.000000,0.000000,0.000000,0.481991
3,0.850459,1.263218,0.930467,0.930467,1.000000,0.963981,0.000000,0.000000,0.000000,0.481991


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,epoch,train_loss,validation_loss,validation_accuracy,precision_0,recall_0,f1_0,precision_1,recall_1,f1_1,validation_macro_f1
0,1.0,1.085094,1.503218,0.930467,0.930467,1.0,0.963981,0.0,0.0,0.0,0.481991
1,2.0,1.082808,1.447987,0.930467,0.930467,1.0,0.963981,0.0,0.0,0.0,0.481991
2,3.0,0.850459,1.263218,0.930467,0.930467,1.0,0.963981,0.0,0.0,0.0,0.481991


In [9]:
# Section 7: Validation Threshold Selection
from sklearn.metrics import precision_recall_fscore_support

val_predictions = trainer.predict(val_tokenized)
val_logits = val_predictions.predictions[0] if isinstance(val_predictions.predictions, tuple) else val_predictions.predictions
assert val_logits.shape == (len(val_df), 2) and np.isfinite(val_logits).all()
val_exp = np.exp(val_logits - np.max(val_logits, axis=1, keepdims=True))
val_probs = val_exp / val_exp.sum(axis=1, keepdims=True)
assert np.isfinite(val_probs).all()
assert np.allclose(val_probs.sum(axis=1), 1.0)
val_dl_prob = val_probs[:, 1]
assert ((val_dl_prob >= 0.0) & (val_dl_prob <= 1.0)).all()

val_labels = val_predictions.label_ids
assert np.array_equal(val_labels, val_df['label'].to_numpy())
assert val_df['candidate_id'].tolist() == AUTHORITATIVE_VAL_CANDIDATE_IDS

threshold_records = []
for tau in np.round(np.arange(0.10, 0.95, 0.05), 2):
    pred = (val_dl_prob >= tau).astype(int)
    _, _, macro_f1, _ = precision_recall_fscore_support(
        val_labels, pred, average='macro', zero_division=0
    )
    _, _, pos_f1, _ = precision_recall_fscore_support(
        val_labels, pred, average='binary', zero_division=0
    )
    threshold_records.append((float(macro_f1), float(pos_f1), float(tau)))

best_macro_f1_val, best_pos_f1_val, best_tau = max(threshold_records, key=lambda x: (x[0], x[1], -x[2]))
FROZEN_BEST_TAU = float(best_tau)
FROZEN_BEST_CKPT = str(best_ckpt)

val_preds_export = pd.DataFrame({
    'candidate_id': val_df['candidate_id'].values,
    'article_id': val_df['article_id'].values,
    'section_id': val_df['section_id'].values,
    'arg1_id': val_df['arg1_id'].values,
    'arg2_id': val_df['arg2_id'].values,
    'true_label': val_labels,
    'dl_prob': val_dl_prob,
    'dl_pred': (val_dl_prob >= FROZEN_BEST_TAU).astype(int),
    'split': 'validation',
    'arg1_before_arg2': val_df['arg1_before_arg2'].values,
})
val_pred_path = DATA_DIR / 'dl_biocause_validation_predictions.csv'
dl_utils.export_predictions(val_preds_export, str(val_pred_path), format_type='csv')

val_verify = pd.read_csv(val_pred_path)
assert len(val_verify) == TARGET_COUNTS['validation']['total']
assert val_verify['candidate_id'].tolist() == AUTHORITATIVE_VAL_CANDIDATE_IDS
assert np.isfinite(val_verify['dl_prob']).all() and val_verify['dl_prob'].between(0.0, 1.0).all()

val_threshold_summary_df = pd.DataFrame([{
    'selected_threshold': FROZEN_BEST_TAU,
    'validation_macro_f1': round(best_macro_f1_val, 4),
    'validation_positive_f1': round(best_pos_f1_val, 4),
    'positive_predictions': int((val_dl_prob >= FROZEN_BEST_TAU).sum()),
    'val_prob_min': round(float(np.min(val_dl_prob)), 4),
    'val_prob_median': round(float(np.median(val_dl_prob)), 4),
    'val_prob_max': round(float(np.max(val_dl_prob)), 4),
}])
val_threshold_summary_df

,selected_threshold,validation_macro_f1,validation_positive_f1,positive_predictions,val_prob_min,val_prob_median,val_prob_max
0,0.1,0.482,0.0,0,0.0047,0.0254,0.0939


In [10]:
# Section 8: Held-Out Test Split Evaluation
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score

if TRAINING_COMPLETED:
    test_path = resolve_split_path('dl_biocause_test.csv')
    if not test_path.exists():
        raise FileNotFoundError('Missing prepared BioCause test split.')
    AUTHORITATIVE_TEST_CANDIDATE_IDS = pd.read_csv(
        test_path, usecols=['candidate_id']
    )['candidate_id'].tolist()
    test_df = pd.read_csv(test_path)
    dl_utils.validate_dataset_schema(test_df, REQUIRED_COLUMNS, dataset_name='BioCause-test')
    assert 'arg1_before_arg2' in test_df.columns
    assert len(test_df) == TARGET_COUNTS['test']['total']
    assert int((test_df['label'] == 1).sum()) == TARGET_COUNTS['test']['positive']
    assert int((test_df['label'] == 0).sum()) == TARGET_COUNTS['test']['negative']
    assert test_df['candidate_id'].is_unique
    assert test_df['candidate_id'].tolist() == AUTHORITATIVE_TEST_CANDIDATE_IDS
    assert set(test_df['article_id']).isdisjoint(set(train_df['article_id']))
    assert set(test_df['article_id']).isdisjoint(set(val_df['article_id']))
    assert set(test_df['label'].unique()) == {0, 1}
    for marker in dl_utils.ENTITY_MARKERS:
        assert test_df['marked_text'].str.contains(re.escape(marker)).all()
    assert all(
        row.find('[E1]') < row.find('[/E1]') and row[row.find('[E1]') + 4:row.find('[/E1]')].strip() != ''
        and row.find('[E2]') < row.find('[/E2]') and row[row.find('[E2]') + 4:row.find('[/E2]')].strip() != ''
        for row in test_df['marked_text']
    ), 'test: empty or malformed entity span'

    test_tokenized = BioCauseDataset(test_df['marked_text'], test_df['label'], tokenizer, MAX_LENGTH)
    assert len(test_tokenized) == TARGET_COUNTS['test']['total']
    assert max(map(len, test_tokenized['input_ids'])) <= MAX_LENGTH
    assert set(test_tokenized['labels']) == {0, 1}
    assert all(all(mid in seq for mid in required_marker_ids) for seq in test_tokenized['input_ids'])
    assert trainer.state.best_model_checkpoint == FROZEN_BEST_CKPT

    test_predictions = trainer.predict(test_tokenized)
    test_logits = test_predictions.predictions[0] if isinstance(test_predictions.predictions, tuple) else test_predictions.predictions
    test_labels = test_predictions.label_ids
    assert test_logits.shape == (len(test_df), 2) and np.isfinite(test_logits).all()
    assert np.array_equal(test_labels, test_df['label'].to_numpy())
    test_exp = np.exp(test_logits - np.max(test_logits, axis=1, keepdims=True))
    test_probs = test_exp / test_exp.sum(axis=1, keepdims=True)
    assert np.isfinite(test_probs).all() and np.allclose(test_probs.sum(axis=1), 1.0)
    test_dl_prob = test_probs[:, 1]
    assert ((test_dl_prob >= 0.0) & (test_dl_prob <= 1.0)).all()

    test_preds = (test_dl_prob >= FROZEN_BEST_TAU).astype(int)
    test_acc = float(np.mean(test_preds == test_labels))
    test_p, test_r, test_f1, _ = precision_recall_fscore_support(
        test_labels, test_preds, average='macro', zero_division=0
    )
    test_pos_p, test_pos_r, test_pos_f1, _ = precision_recall_fscore_support(
        test_labels, test_preds, average='binary', zero_division=0
    )
    test_roc_auc = float(roc_auc_score(test_labels, test_dl_prob))
    test_pr_auc = float(average_precision_score(test_labels, test_dl_prob))
    cm = confusion_matrix(test_labels, test_preds, labels=[0, 1])

    grouped_order_metrics = {}
    for order_val, group_idx in test_df.groupby('arg1_before_arg2').groups.items():
        idx = list(group_idx)
        sub_y = test_labels[idx]
        sub_p = test_preds[idx]
        _, _, sub_macro_f1, _ = precision_recall_fscore_support(sub_y, sub_p, average='macro', zero_division=0)
        gp_p, gp_r, gp_f1, _ = precision_recall_fscore_support(sub_y, sub_p, average='binary', zero_division=0)
        key = 'arg1_before_arg2' if bool(order_val) else 'arg2_before_arg1'
        grouped_order_metrics[key] = {
            'n_examples': int(len(idx)),
            'n_positive': int((sub_y == 1).sum()),
            'accuracy': round(float(np.mean(sub_p == sub_y)), 4),
            'positive_precision': round(float(gp_p), 4),
            'positive_recall': round(float(gp_r), 4),
            'positive_f1': round(float(gp_f1), 4),
            'macro_f1': round(float(sub_macro_f1), 4),
        }

    grouped_article_metrics = {}
    for art_id, group_idx in test_df.groupby('article_id').groups.items():
        idx = list(group_idx)
        sub_y = test_labels[idx]
        sub_p = test_preds[idx]
        _, _, sub_macro_f1, _ = precision_recall_fscore_support(sub_y, sub_p, average='macro', zero_division=0)
        gp_p, gp_r, gp_f1, _ = precision_recall_fscore_support(sub_y, sub_p, average='binary', zero_division=0)
        grouped_article_metrics[str(art_id)] = {
            'n_examples': int(len(idx)),
            'n_positive': int((sub_y == 1).sum()),
            'accuracy': round(float(np.mean(sub_p == sub_y)), 4),
            'positive_precision': round(float(gp_p), 4),
            'positive_recall': round(float(gp_r), 4),
            'positive_f1': round(float(gp_f1), 4),
            'macro_f1': round(float(sub_macro_f1), 4),
        }

    test_summary_df = pd.DataFrame([{
        'split': 'test',
        'threshold': FROZEN_BEST_TAU,
        'n_examples': int(len(test_labels)),
        'positive_predictions': int((test_preds == 1).sum()),
        'accuracy': round(test_acc, 4),
        'macro_precision': round(float(test_p), 4),
        'macro_recall': round(float(test_r), 4),
        'macro_f1': round(float(test_f1), 4),
        'positive_precision': round(float(test_pos_p), 4),
        'positive_recall': round(float(test_pos_r), 4),
        'positive_f1': round(float(test_pos_f1), 4),
        'roc_auc': round(test_roc_auc, 4),
        'pr_auc': round(test_pr_auc, 4),
    }])
else:
    test_df = None
    test_tokenized = None
    test_acc = test_p = test_r = test_f1 = None
    test_pos_p = test_pos_r = test_pos_f1 = test_roc_auc = test_pr_auc = None
    cm = np.zeros((2, 2), dtype=int)
    grouped_order_metrics = {}
    grouped_article_metrics = {}
    test_summary_df = pd.DataFrame()

test_summary_df

,split,threshold,n_examples,positive_predictions,accuracy,macro_precision,macro_recall,macro_f1,positive_precision,positive_recall,positive_f1,roc_auc,pr_auc
0,test,0.1,1394,0,0.9283,0.4641,0.5,0.4814,0.0,0.0,0.0,0.7877,0.2218


In [11]:
# Section 9: Prediction & Metrics Exports & Model Persistence
EXPORT_DIR = ROOT_DIR / 'data' / 'processed'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if TRAINING_COMPLETED and (test_df is not None):
    # 1. Export test predictions DataFrame (standardized schema matching validation)
    predictions_export = pd.DataFrame({
        'candidate_id': test_df['candidate_id'].values,
        'article_id': test_df['article_id'].values,
        'section_id': test_df['section_id'].values,
        'arg1_id': test_df['arg1_id'].values,
        'arg2_id': test_df['arg2_id'].values,
        'true_label': test_labels,
        'dl_prob': test_dl_prob,
        'dl_pred': test_preds,
        'split': 'test',
        'arg1_before_arg2': test_df['arg1_before_arg2'].values,
    })

    pred_path = EXPORT_DIR / 'dl_biocause_test_predictions.csv'
    assert predictions_export['candidate_id'].tolist() == AUTHORITATIVE_TEST_CANDIDATE_IDS
    dl_utils.export_predictions(predictions_export, str(pred_path), format_type='csv')

    pred_verify = pd.read_csv(pred_path)
    REQUIRED_PRED_COLS = [
        'candidate_id', 'article_id', 'section_id', 'arg1_id', 'arg2_id',
        'true_label', 'dl_prob', 'dl_pred', 'split', 'arg1_before_arg2'
    ]
    assert len(pred_verify) == 1394, f'Expected 1394 rows, got {len(pred_verify)}'
    assert list(pred_verify.columns) == REQUIRED_PRED_COLS, f'Column mismatch: {list(pred_verify.columns)}'
    for col in REQUIRED_PRED_COLS:
        assert pred_verify[col].isna().sum() == 0, f'Found NaN in {col}'
    assert set(pred_verify['true_label'].unique()).issubset({0, 1}), 'Non-binary true_label'
    assert set(pred_verify['dl_pred'].unique()).issubset({0, 1}), 'Non-binary dl_pred'
    assert np.isfinite(pred_verify['dl_prob']).all(), 'Non-finite dl_prob'
    assert pred_verify['dl_prob'].between(0.0, 1.0).all(), 'Probabilities out of [0, 1] range'
    assert pred_verify['candidate_id'].tolist() == AUTHORITATIVE_TEST_CANDIDATE_IDS
    assert (pred_verify['true_label'].values == test_df['label'].values).all()
    assert (pred_verify['split'] == 'test').all()

    # Cross-split schema compatibility assertions
    val_pred_path = EXPORT_DIR / 'dl_biocause_validation_predictions.csv'
    assert val_pred_path.exists(), 'Missing validation prediction export'
    val_check = pd.read_csv(val_pred_path)
    assert val_check['candidate_id'].tolist() == AUTHORITATIVE_VAL_CANDIDATE_IDS
    assert list(val_check.columns) == list(pred_verify.columns), (
        f'Schema mismatch: val {list(val_check.columns)} vs test {list(pred_verify.columns)}'
    )
    for col in REQUIRED_PRED_COLS:
        assert val_check[col].dtype == pred_verify[col].dtype, f'Dtype mismatch in {col}'

    # 2. Export compact metrics JSON
    metrics_export = {
        'task': 'BioCause PubMedBERT Fine-Tuning',
        'model': PUBMEDBERT_CHECKPOINT,
        'training_completed': True,
        'decision_threshold': float(FROZEN_BEST_TAU),
        'split_counts': {
            'train': {'total': len(train_df), 'positive': int((train_df['label'] == 1).sum()), 'negative': int((train_df['label'] == 0).sum())},
            'validation': {'total': len(val_df), 'positive': int((val_df['label'] == 1).sum()), 'negative': int((val_df['label'] == 0).sum())},
            'test': {'total': len(test_df), 'positive': int((test_df['label'] == 1).sum()), 'negative': int((test_df['label'] == 0).sum())},
        },
        'hyperparameters': {
            'num_train_epochs': NUM_EPOCHS,
            'batch_size': BATCH_SIZE,
            'eval_batch_size': EVAL_BATCH_SIZE,
            'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
            'learning_rate': LEARNING_RATE,
            'weight_decay': WEIGHT_DECAY,
            'max_length': MAX_LENGTH,
            'fp16': FP16_ENABLED,
            'seed': SEED,
            'pos_weight': float(pos_weight),
        },
        'best_checkpoint': FROZEN_BEST_CKPT,
        'best_val_macro_f1': float(best_metric) if isinstance(best_metric, (int, float)) else None,
        'epoch_metrics': epoch_metrics,
        'test_overall': {
            'n_examples': int(len(test_labels)),
            'positive_predictions': int((test_preds == 1).sum()),
            'accuracy': round(float(test_acc), 4),
            'macro_precision': round(float(test_p), 4),
            'macro_recall': round(float(test_r), 4),
            'macro_f1': round(float(test_f1), 4),
            'positive_precision': round(float(test_pos_p), 4),
            'positive_recall': round(float(test_pos_r), 4),
            'positive_f1': round(float(test_pos_f1), 4),
            'roc_auc': round(float(test_roc_auc), 4),
            'pr_auc': round(float(test_pr_auc), 4),
        },
        'confusion_matrix': cm.tolist(),
        'test_by_argument_order': grouped_order_metrics,
        'test_by_article': grouped_article_metrics,
    }

    metrics_path = EXPORT_DIR / 'dl_biocause_metrics.json'
    with open(metrics_path, 'w') as f:
        json.dump(metrics_export, f, indent=2, default=lambda x: x.item() if hasattr(x, 'item') else str(x))
    assert metrics_path.exists(), f'Missing metrics export: {metrics_path.name}'

    # 3. Save best trained model weights and tokenizer
    BEST_MODEL_DIR = ROOT_DIR / 'DL' / 'models' / 'biocause_pubmedbert_best'
    BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(BEST_MODEL_DIR))
    tokenizer.save_pretrained(str(BEST_MODEL_DIR))
    assert (BEST_MODEL_DIR / 'config.json').exists(), 'Saved model config.json missing'

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 5. Expected Outputs & Evaluation Criteria

- Best fine-tuned PubMedBERT checkpoint.
- `data/processed/dl_biocause_validation_predictions.csv`: 1,884 validation candidates with raw probabilities and decisions.
- `data/processed/dl_biocause_test_predictions.csv`: 1,394 test candidates with raw probabilities and decisions.
- **Evaluation Criteria**: Candidate-level Precision, Recall, and F1 on the 4 held-out test publications (100 gold relations, 1,394 candidates).
